# Skin CD4 malignancy via same-sample CD8 inferCNV reference

Benchmark variant of `14_skin_T_tcr_cnv_malignancy.ipynb`. CTCL/MF is overwhelmingly a **CD4** malignancy, so same-sample **CD8** T cells are a clean, lineage-matched benign diploid reference.

- **Exclude** samples whose malignant `lineage` is `CD8` or `gamma_delta`; keep `CD4` / `not_applicable` / `unresolved_provisional`.
- Per kept sample: inferCNV with **same-sample CD8 as reference**, **CD4 as query**; call malignancy on CD4.
- Sweep `(CNV_LEIDEN_RES × CLUSTER_THR_SCALE)` to maximize F1 vs the orthogonal TCR dominant-clone call — same machinery as nb14.

> Heavy cells (per-sample inferCNV) marked **HEAVY — run on GPU kernel**.

In [ ]:
import sys, gc
from pathlib import Path
import numpy as np
import pandas as pd
import scanpy as sc


def _resolve_nb_dir() -> Path:
    start = Path.cwd()
    for base in [start, *start.parents]:
        for sub in [Path("."), Path("notebooks/MF"), Path("scvi-tools-neural-nmf/notebooks/MF")]:
            cand = base / sub
            if cand.name == "MF" and (cand / "data").exists():
                return cand.resolve()
    raise FileNotFoundError(start)


NB_DIR = _resolve_nb_dir()
if str(NB_DIR) not in sys.path:
    sys.path.insert(0, str(NB_DIR))
import importlib
import atlas_join_helpers as H
import skin_T_cnv_helpers as C            # shared with nb14
importlib.reload(H); importlib.reload(C)

SEED = 0
np.random.seed(SEED)
sc.settings.verbosity = 1

V2          = NB_DIR / "data" / "atlas_joint" / "skin_T_tcr_malig_v2.h5ad"  # curated TCR-complete object
GTF         = NB_DIR / "data" / "cache" / "Homo_sapiens.GRCh38.110.chr.gtf.gz"
UMAP_NPY    = NB_DIR / "data" / "atlas_joint" / "skin_T_umap_u.npy"         # shared MRVI u-UMAP (Step 9)
OUT_PARQUET = NB_DIR / "data" / "atlas_joint" / "skin_cd4_cd8ref_malignancy.parquet"
FIG_DIR     = NB_DIR / "figures"; FIG_DIR.mkdir(exist_ok=True)

## Step 1–2 — load the curated TCR-complete skin-T object

Loads `skin_T_tcr_malig_v2.h5ad` (242,959 cells, 34 kept donors). The upstream ALICE tumor call
`tcr_malignant_alice` is carried in as `tcr_is_malignant` (and `tcr_is_dominant_clone`, so it also
defines the benign/diploid floor in Step 6); `cell_type_T2` drives the CD4-query / CD8-reference
split in Step 4. `tcr_is_expanded` / `has_tcr` / `tcr_clone_*` / `cached_malignant` come from the file.

In [ ]:
adata = sc.read_h5ad(V2)                       # curated TCR-complete T object (242,959 cells)
# ALICE tumor call is the malignant truth; it also drives benign/reference exclusion downstream
alice = adata.obs["tcr_malignant_alice"].astype(bool).to_numpy()
adata.obs["tcr_is_malignant"]      = alice
adata.obs["tcr_is_dominant_clone"] = alice
# tcr_is_expanded / has_tcr / tcr_clone_size / tcr_clone_id / cached_malignant come from the file
print(adata.shape, "| alice malignant:", int(alice.sum()))

## Step 3 — carried TCR malignancy (ALICE)

The malignant call is the upstream ALICE tumor annotation (`tcr_malignant_alice`), carried in as
`tcr_is_malignant`. Sanity-check its distribution by `cell_type_T2` (CD4-restricted) and by study.

In [ ]:
# carried ALICE malignancy — distribution by cell_type_T2 and by study
print("tcr_is_malignant by cell_type_T2:\n",
      adata.obs.groupby("cell_type_T2", observed=True)["tcr_is_malignant"].agg(["size", "sum"]))
print("\ntcr_is_malignant by study:\n",
      adata.obs.groupby("study", observed=True)["tcr_is_malignant"].agg(["size", "sum"]))

## Step 4 — sample filter (drop CD8/γδ) + CD8-reference inferCNV input prep  · HEAVY (GPU kernel)

Keep samples with `lineage ∈ {CD4, not_applicable, unresolved_provisional}` (drop CD8 / gamma_delta MF, where CD8 is not a benign reference). Within kept samples split cells by `cell_type_T` prefix: **CD4_\*** = query, **CD8_\*** = reference (`UNK` dropped). Samples with `< MIN_CD8_REF` CD8 cells are dropped (no usable reference).

In [ ]:
import infercnvpy as cnv
import anndata as ad

KEPT_LINEAGES = ["CD4", "not_applicable", "unresolved_provisional"]
MIN_CD8_REF   = 20
STD_CHR       = [f"chr{c}" for c in list(range(1, 23)) + ["X", "Y"]]

# CD4 query + same-sample CD8 reference, log-normalised, positioned, std-chr only (drops chrnan/chrM)
acnv, CNV_DONORS = C.prepare_cd4_cd8ref_inputs(
    adata, GTF, kept_lineages=KEPT_LINEAGES, min_cd8_ref=MIN_CD8_REF, std_chr=STD_CHR)

## Step 5 — per-sample inferCNV with CD8 reference  · HEAVY (GPU kernel, long-running)

Per sample, `cnv.tl.infercnv` with `reference_cat=["cd8_ref"]`, then cluster + score the **CD4 query** cells only at every `RES_GRID` resolution (one neighbor graph, Leiden per resolution). `cnv_score` is the per-cluster mean of the per-cell `cnv_cell_score`, recomputed analytically so the Step-6f sweep needs no inferCNV recompute. Cached to `skin_cd4_cd8ref_cnv_per_cell_multires.parquet`.

In [ ]:
WINDOW_SIZE    = 250          # inferCNV smoothing window (fixed default)
FORCE_CNV      = True         # recompute on the v2 cell set (old cache is 401k); set False after 1st run
CNV_LEIDEN_RES = 1.0          # >>> KNOB 1: CNV-cluster Leiden resolution — change & rerun Step 5 <<<
CNV_N_JOBS     = 8            # cap workers (infercnvpy forks one worker per chunk -> OOM if uncapped)
CNV_CHUNK      = 2500
# 4-strategy knobs (Step 6): high-res MRVI-Leiden smoothing (strat2 & strat4)
MRVI_LEIDEN_RES = 2.0         # high resolution -> fine, clone-pure clusters
MRVI_CLUST_FRAC = 0.8         # MRVI cluster malignant if >= this frac of its cells are called
# resolution-tagged cache: changing CNV_LEIDEN_RES recomputes into its own file (old kept)
CNV_CACHE = NB_DIR / "data" / "atlas_joint" / f"skin_cd4_cd8ref_cnv_per_cell_res{CNV_LEIDEN_RES:g}.parquet"

C.run_per_sample_cd8ref_infercnv(acnv, CNV_DONORS, CNV_CACHE, window=WINDOW_SIZE,
                                 leiden_res=CNV_LEIDEN_RES, n_jobs=CNV_N_JOBS, chunk=CNV_CHUNK,
                                 std_chr=STD_CHR, force=FORCE_CNV)

## Step 6 — base per-cell call (strat1) + shared MRVI clustering  · HEAVY (GPU)

Computes the pieces the 4 strategies share (same machinery as nb14, via `skin_T_cnv_helpers`):
- **diploid floor** = benign CD4 query cells (non-dominant-clone, non-tumor).
- **strategy 1** (`cnv_malignant`) — per-cell 2-component GMM crossover on `log(cnv_cell_score)` with the diploid-median floor (`C.call_per_donor`). CD8 reference cells carry no score → excluded automatically.
- **MRVI-Leiden clustering** of the CD4 query on the `X_mrvi_u` latent — computed **once** here (heavy) and reused by strat2 / strat4 in Step 6b, so retuning the threshold stays cheap.

In [ ]:
# benign/diploid floor = benign CD4 query cells (non-dominant-clone, non-tumor). The CD8 cells
# define the inferCNV baseline upstream (Step 5), not this floor.
diploid = ((acnv.obs["cnv_ref"] == "query")
           & ~acnv.obs["tcr_is_dominant_clone"].to_numpy()
           & (acnv.obs["cell_type"].astype(str) != "tumor_cell"))

# strategy 1 — per-cell GMM crossover on the genome-wide cnv_cell_score (C.call_per_donor, reused from nb14)
call1, meth1 = C.call_per_donor("cnv_cell_score", diploid, CNV_DONORS, acnv.obs, seed=SEED)
acnv.obs["cnv_malignant"]   = call1
acnv.obs["cnv_call_method"] = meth1

# high-res MRVI-Leiden on the CD4 query only (CD8 ref excluded) — shared by strat2 & strat4.  HEAVY (GPU)
acnvq = acnv[acnv.obs["cnv_ref"] == "query"].copy()
C.compute_mrvi_leiden(acnvq, MRVI_LEIDEN_RES, SEED)
acnv.obs["mrvi_leiden"] = "ref"                                  # CD8 ref cells -> own group (never voted malignant)
acnv.obs.loc[acnvq.obs_names, "mrvi_leiden"] = acnvq.obs["mrvi_leiden"].astype(str).values
del acnvq; gc.collect()

In [ ]:
# propagate CD4 CNV scores + cluster labels onto the full object obs (for the benchmark / Step 8)
adata.obs["cnv_ref"]        = acnv.obs["cnv_ref"].reindex(adata.obs_names).fillna("")
adata.obs["cnv_leiden"]     = acnv.obs["cnv_leiden"].reindex(adata.obs_names).fillna("")
adata.obs["cnv_cell_score"] = acnv.obs["cnv_cell_score"].reindex(adata.obs_names)
adata.obs["cnv_score"]      = acnv.obs["cnv_score"].reindex(adata.obs_names)

## Step 6b — assemble the 4 CNV strategies + threshold knob

Builds the same 4 methods as nb14 and exposes the cheap threshold knob:
- **strat1 cell-GMM** (`cnv_malignant`) — per-cell GMM on `cnv_cell_score` (Step 6).
- **strat2 cell-MRVI** (`cnv_malig_leiden_mrvi`) — strat1 majority-voted over MRVI-Leiden clusters.
- **strat3 cnv-cluster** (`cnv_malig_cluster`) — per-cnv-cluster GMM on `cnv_score`.
- **strat4 cnv-cluster-MRVI** (`cnv_malig_cluster_mrvi`) — strat3 majority-voted over MRVI clusters.

`CLUSTER_THR_SCALE` (`< 1` → more cells malignant) is cheap to retune — rerun this cell only (the heavy MRVI clustering from Step 6 is reused).

In [ ]:
# >>> KNOB 2: per-sample threshold scale (smaller -> more cells malignant). Rerun this cell to change. <<<
CLUSTER_THR_SCALE = 0.85

# strategy 3 — per-cnv-cluster GMM on cnv_score
call3, _ = C.call_per_donor("cnv_score", diploid, CNV_DONORS, acnv.obs,
                            verbose=False, seed=SEED, thr_scale=CLUSTER_THR_SCALE)
acnv.obs["cnv_malig_cluster"] = call3

# strategies 2 & 4 — MRVI-cluster majority vote of strat1 / strat3 (cheap; reuses cached mrvi_leiden)
acnv.obs["cnv_malig_leiden_mrvi"]  = C.vote_cluster(acnv, "cnv_malignant", MRVI_CLUST_FRAC)
acnv.obs["cnv_malig_cluster_mrvi"] = C.vote_cluster(acnv, "cnv_malig_cluster", MRVI_CLUST_FRAC)

METHODS = {"strat1_cell_gmm":         "cnv_malignant",
           "strat2_cell_mrvi":        "cnv_malig_leiden_mrvi",
           "strat3_cnv_cluster":      "cnv_malig_cluster",
           "strat4_cnv_cluster_mrvi": "cnv_malig_cluster_mrvi"}

# propagate the call method + 4 calls onto the full object (False outside the CD4 query)
adata.obs["cnv_call_method"] = acnv.obs["cnv_call_method"].reindex(adata.obs_names).fillna("")
for col in METHODS.values():
    adata.obs[col] = acnv.obs[col].reindex(adata.obs_names).fillna(False).astype(bool)

n_q = int((adata.obs["cnv_ref"] == "query").sum())
print(f"strategy calls (res={CNV_LEIDEN_RES:g}, thr_scale={CLUSTER_THR_SCALE}) on {n_q} CD4 query cells:")
for name, col in METHODS.items():
    print(f"  {name:24s} {col:24s} malignant={int(adata.obs[col].sum())}/{n_q}")

## Step 6d — benchmark the 4 CD4 CNV strategies vs TCR

Precision / recall / F1 / Jaccard of each CNV strategy (S1 cell-GMM, S2 cell-MRVI, S3 cnv-cluster, S4 cnv-cluster-MRVI) against the orthogonal TCR dominant-clone call, on TCR+ CD4 cells — same `C.strategy_agreement` machinery as nb14.

In [ ]:
import matplotlib.pyplot as plt

# agreement of each CNV strategy with the orthogonal TCR dominant-clone call on TCR+ CD4 cells
avail = adata.obs["has_tcr"].to_numpy() & (adata.obs["cnv_ref"].to_numpy() == "query")
agree = C.strategy_agreement(adata, METHODS, "tcr_is_malignant", avail)
print(f"agreement vs TCR on {int(avail.sum())} TCR+ CD4 cells:")
print(agree.to_string())

print("\nstrat3_cnv_cluster × TCR:\n",
      pd.crosstab(adata.obs.loc[avail, "tcr_is_malignant"],
                  adata.obs.loc[avail, "cnv_malig_cluster"], rownames=["tcr"], colnames=["cnv_cluster"]))

# malignant fraction by study / sample: TCR vs the 4 methods
rename = {"tcr_is_malignant": "TCR", "cnv_malignant": "S1 cell-GMM",
          "cnv_malig_leiden_mrvi": "S2 cell-MRVI", "cnv_malig_cluster": "S3 cnv-cluster",
          "cnv_malig_cluster_mrvi": "S4 cnv-cluster-MRVI"}
frac_cols = ["tcr_is_malignant", *METHODS.values()]
q = adata.obs[adata.obs["sample_id"].isin(CNV_DONORS) & (adata.obs["cnv_ref"] == "query")]
for by, fs in [("study", (6, 3)), ("sample_id", (12, 3))]:
    tbl = q.groupby(by, observed=True)[frac_cols].mean().rename(columns=rename)
    ax = tbl.plot.bar(figsize=fs)
    ax.set_ylabel("malignant fraction"); ax.set_xlabel("")
    ax.set_title(f"CD4 CNV methods vs TCR — malignant fraction by {by}")
    plt.tight_layout(); plt.savefig(FIG_DIR / f"skin_cd4_cd8ref_frac_by_{by}.png", dpi=150); plt.show()

## Step 6e — inferCNV chromosome heatmaps (annotated) for representative samples  · HEAVY (GPU)

`C.run_cnv_heatmaps` (shared with nb14, `shared_ref=None` since the same-sample CD8 reference already lives in `acnv`) re-runs inferCNV for a few donors keeping `X_cnv`, then draws a chromosome heatmap with three per-cell color strips on the right: the chosen strategy's call (`HEATMAP_METHOD`), **malignant TCR clone** membership, and **TCR clone size** (log).

`ORDER_BY` sets the row grouping — two different clusterings:
- `"cnv_leiden"` (default) — the cached Step-5 cluster the per-cnv-cluster strategy calls on, so the call strip is **constant within each block**.
- `"recompute"` — a fresh Leiden on this run's (`window=150`) `X_cnv`: visually tighter blocks, but the cached-clustering call strip will be mixed within them.

Color knobs: `HM_VLIM_SCALE` (higher → paler), `HM_CMAP`, `HM_WINDOW`.

In [ ]:
# inferCNV chromosome heatmaps for representative samples (N_PER_STUDY per study, top TCR burden),
# annotated with the chosen strategy's call + malignant-TCR-clone membership + clone size.
# Reuses C.run_cnv_heatmaps with shared_ref=None (same-sample CD8 reference is already inside acnv)
# and reference_cat=("cd8_ref",).  HEAVY (GPU kernel).
#
# ORDER_BY sets the row grouping (two DIFFERENT clusterings):
#   "cnv_leiden" (default) — cached Step-5 cnv_leiden, the unit strategy 3 calls on, so the
#                            chosen-call strip is CONSTANT within every block.
#   "recompute"            — fresh leiden on THIS run's X_cnv: tighter blocks, but the strat strip
#                            (made on the cached clustering) will be mixed within them.
RUN_CNV_HEATMAP = True
HEATMAP_METHOD  = "strat3_cnv_cluster"   # any key of METHODS
ORDER_BY        = "cnv_leiden"           # "cnv_leiden" (strip uniform per block) | "recompute"
HM_WINDOW       = 150                    # smooth out per-cell speckle so blocks read as blocks
HM_VLIM_SCALE   = 3.0                    # widen color range: diploid -> near-white, only real CNV shows color
HM_CMAP         = "RdBu_r"
N_PER_STUDY     = 3

if RUN_CNV_HEATMAP:
    C.run_cnv_heatmaps(acnv, None, call_col=METHODS[HEATMAP_METHOD], fig_dir=FIG_DIR,
                       reference_cat=("cd8_ref",), fig_prefix="skin_cd4_cd8ref_cnv_heatmap",
                       n_per_study=N_PER_STUDY, window=HM_WINDOW, vlim_scale=HM_VLIM_SCALE,
                       cmap=HM_CMAP, order_by=ORDER_BY, n_jobs=CNV_N_JOBS, chunk=CNV_CHUNK)
else:
    print("RUN_CNV_HEATMAP=False — skipping the heavy per-donor heatmap recompute")

## Step 7 — combine TCR + CD4 CNV call & report

In [ ]:
# Final CD4 CNV call = per-cnv-cluster strategy 3. Combine with TCR.
tcr_m = adata.obs["tcr_is_malignant"].to_numpy()
cnv_m = adata.obs["cnv_malig_cluster"].to_numpy().astype(bool)
adata.obs["combined_malignant"] = tcr_m | cnv_m
adata.obs["malignant_evidence"] = np.select(
    [tcr_m & cnv_m, tcr_m & ~cnv_m, ~tcr_m & cnv_m],
    ["both", "tcr_only", "cnv_only"], default="none")
print("combined malignant:", int(adata.obs["combined_malignant"].sum()), "/", adata.n_obs)
print("\nevidence:\n", adata.obs["malignant_evidence"].value_counts())

both_avail = adata.obs["has_tcr"].to_numpy() & (adata.obs["cnv_ref"].to_numpy() == "query")
print("\nTCR x CNV crosstab (TCR+ CD4 cells):")
print(pd.crosstab(adata.obs.loc[both_avail, "tcr_is_malignant"],
                  adata.obs.loc[both_avail, "cnv_malig_cluster"],
                  rownames=["tcr"], colnames=["cnv_cluster"]))

## Step 8 — persist CD4 obs annotations

In [ ]:
keep = ["donor", "sample_id", "study", "disease", "lineage", "cell_type", "cell_type_T",
        "cnv_ref", "cached_malignant", "has_tcr", "tcr_clone_id", "tcr_clone_size",
        "tcr_is_expanded", "tcr_is_dominant_clone", "tcr_is_malignant",
        "cnv_score", "cnv_cell_score", "cnv_leiden", "cnv_call_method",
        "cnv_malignant", "cnv_malig_leiden_mrvi", "cnv_malig_cluster", "cnv_malig_cluster_mrvi",
        "combined_malignant", "malignant_evidence"]
out = adata.obs.loc[adata.obs["cnv_ref"] == "query", [c for c in keep if c in adata.obs.columns]].copy()
out.index.name = "obs_name"
out.to_parquet(OUT_PARQUET)
print("wrote", OUT_PARQUET, out.shape)
out.head()

## Step 9 — UMAP: TCR + the 4 CD4 CNV methods

One panel per call on the shared MRVI u-UMAP (`X_mrvi_u`-derived), same as nb14 Step 8. The TCR
panel uses `tcr_label` — the **same malignant-clone scheme as nb16/17** (malignant = ALICE clone,
benign = TCR⁺ non-malignant unexpanded, else unlabeled) — so the malignant set and the benign
reference match across notebooks. CNV calls are defined only on the CD4 query (CD8 / NK appear benign).

In [ ]:
if UMAP_NPY.exists() and np.load(UMAP_NPY).shape[0] == adata.n_obs:
    adata.obsm["X_umap"] = np.load(UMAP_NPY)
else:                                          # stale / missing (cell set changed) -> recompute on X_mrvi_u
    sc.pp.neighbors(adata, use_rep="X_mrvi_u", random_state=SEED)
    sc.tl.umap(adata, random_state=SEED)
    np.save(UMAP_NPY, adata.obsm["X_umap"])
    print("recomputed u-UMAP ->", UMAP_NPY.name)
# tcr_label = identical malignant-clone scheme as nb16/17: malignant = ALICE clone,
# benign = TCR+ non-malignant unexpanded, else unlabeled (so the reference set matches too).
has = adata.obs["has_tcr"].astype(bool).to_numpy()
mal = adata.obs["tcr_is_malignant"].astype(bool).to_numpy()
exp = adata.obs["tcr_is_expanded"].astype(bool).to_numpy()
adata.obs["tcr_label"] = pd.Categorical(
    np.where(mal, "malignant", np.where(has & ~mal & ~exp, "benign", "unlabeled")),
    categories=["malignant", "benign", "unlabeled"])
adata.uns["tcr_label_colors"] = ["#d62728", "#1f77b4", "#dddddd"]   # malignant / benign / unlabeled
for c in METHODS.values():
    adata.obs[c + "_lbl"] = (adata.obs[c].astype(bool)
                             .map({True: "malignant", False: "benign"}).astype("category"))
sc.pl.umap(adata, color=["tcr_label", *[c + "_lbl" for c in METHODS.values()]],
           ncols=2, size=2, wspace=0.3, save="_skin_cd4_cd8ref_cnv_methods.png")

## Step 10 — TCR↔CNV quality metrics

For each method, restricted to TCR+ CD4 cells, pooled agreement with the TCR call:
**sensitivity** = of TCR-malignant cells, fraction called CNV-malignant; **specificity** =
of TCR-non-malignant cells, fraction called CNV-non-malignant (with raw counts). Same
`C.tcr_cnv_quality` machinery as nb14.

In [ ]:
# Step 10 — TCR<->CNV quality metrics (TCR+ CD4 cells, per method)
avail = adata.obs["has_tcr"].to_numpy() & (adata.obs["cnv_ref"].to_numpy() == "query")
quality = C.tcr_cnv_quality(adata, METHODS, "tcr_is_malignant", avail)
print(f"TCR<->CNV quality on {int(avail.sum())} TCR+ CD4 cells:")
print(quality.to_string())